# Semana 9 — Limpieza e ingeniería de variables
Se usan enero-febrero 2026 para entrenamiento y marzo para validación. La duración se calcula únicamente para crear el objetivo, nunca como predictor. Se excluyen tarifas, distancia real y fecha de llegada.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src.data import download_yellow_taxi_months, load_month
from src.modeling import prepare, sample_month, FEATURES, models, metrics
from sklearn.model_selection import ParameterGrid
import json
SEED=42
MAX_ROWS_PER_MONTH=100000  # Cambiar a None para usar todos los registros (requiere memoria)
MONTHS=["2026-01","2026-02","2026-03"]
# Descarga automática; requiere conexión y espacio en disco
paths=download_yellow_taxi_months(MONTHS, ROOT/'data')
raw={m:load_month(p,columns=["tpep_pickup_datetime","tpep_dropoff_datetime","passenger_count","PULocationID","DOLocationID"]) for m,p in paths.items()}
frames={m:sample_month(prepare(df),MAX_ROWS_PER_MONTH,SEED) for m,df in raw.items()}
train=pd.concat([frames["2026-01"],frames["2026-02"]],ignore_index=True)
val=frames["2026-03"]
X_train,y_train=train[FEATURES],train["duration_min"]
X_val,y_val=val[FEATURES],val["duration_min"]
print('Train:',len(train),'Validación:',len(val))

In [ ]:
print(train[FEATURES+["duration_min"]].isna().mean().sort_values(ascending=False))
print(train["duration_min"].describe(percentiles=[.1,.5,.9,.95,.99]))
print('Variables:',FEATURES)
print('Meses de entrenamiento:', sorted(train.tpep_pickup_datetime.dt.month.unique()))
print('Mes de validación:', sorted(val.tpep_pickup_datetime.dt.month.unique()))

**Decisiones:** duraciones entre 1 y 180 minutos; zonas faltantes descartadas; variables numéricas imputadas con mediana; zonas codificadas con one-hot ajustado exclusivamente al entrenamiento. La muestra aleatoria por mes reduce costo de cómputo, sin mezclar meses entre particiones.